In [1]:
import torch

# Relationship is y = x ^ 2
x = torch.tensor([[-3.0],
                  [-2.0],
                  [-1.0],
                  [ 0.0],
                  [ 1.0],
                  [ 2.0],
                  [ 3.0]])

y = torch.tensor([[9.0],
                  [4.0],
                  [1.0],
                  [0.0],
                  [1.0],
                  [4.0],
                  [9.0]])

 #; 
model = torch.nn.Sequential(
    #makes 1 input into 8 different linear equations (y=mx+b);
    torch.nn.Linear(1, 8),
    #makes negative answers 0
    torch.nn.ReLU(), 
    #then takes the 8 predictions and puts it in one equation for one output
        #EndPrediction=(h1×weight[0]) + (h2×weight[1]) + (h3×weight[2]) + (⋯) + (h8×weight[7]) + bias
    torch.nn.Linear(8, 1)
)

In [2]:
print(model[0].weight)
print(model[0].bias)

Parameter containing:
tensor([[ 0.9820],
        [-0.8244],
        [ 0.0445],
        [-0.7215],
        [ 0.5052],
        [-0.2063],
        [ 0.4680],
        [ 0.7226]], requires_grad=True)
Parameter containing:
tensor([-0.6428,  0.7885,  0.3218, -0.8836,  0.7689,  0.1701,  0.1311, -0.1713],
       requires_grad=True)


In [3]:
test_x = torch.tensor([[2.0]])

hidden_linear = model[0](test_x)
print("Before ReLU:")
print(hidden_linear)

Before ReLU:
tensor([[ 1.3213, -0.8604,  0.4107, -2.3266,  1.7793, -0.2425,  1.0672,  1.2740]],
       grad_fn=<AddmmBackward0>)


In [4]:
hidden_output = model[1](hidden_linear)
print("After ReLU:")
print(hidden_output)

After ReLU:
tensor([[1.3213, 0.0000, 0.4107, 0.0000, 1.7793, 0.0000, 1.0672, 1.2740]],
       grad_fn=<ReluBackward0>)


In [5]:
prediction = model[2](hidden_output)
print("Final prediction:")
print(prediction)

Final prediction:
tensor([[-0.5760]], grad_fn=<AddmmBackward0>)


In [6]:
#Measure difference squared
loss_fn = torch.nn.MSELoss()

In [7]:
#Make the oprimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


In [19]:
prediction = model(x)
loss = loss_fn(prediction, y)

print("Prediction:")
print(prediction)

print("Loss:")
print(loss)

Prediction:
tensor([[ 0.2717],
        [ 0.2730],
        [ 0.2146],
        [ 0.0678],
        [-0.1918],
        [-0.5760],
        [-0.9602]], grad_fn=<AddmmBackward0>)
Loss:
tensor(30.3231, grad_fn=<MseLossBackward0>)


In [20]:
#reset gradiants
optimizer.zero_grad()

#make gradiants
loss.backward()

#print gradiants 1-8
print("Hidden layer weight gradients:")
print(model[0].weight.grad)

print("\nHidden layer bias gradients:")
print(model[0].bias.grad)

#print 8-1 gradiants
print("\nOutput layer weight gradients:")
print(model[2].weight.grad)

print("\nOutput layer bias gradient:")
print(model[2].bias.grad)

Hidden layer weight gradients:
tensor([[ 2.2153],
        [-0.8328],
        [ 0.0525],
        [-0.0513],
        [ 2.5139],
        [ 3.3815],
        [ 2.7684],
        [-0.5079]])

Hidden layer bias gradients:
tensor([ 0.8662,  0.3187,  0.2616,  0.0190,  1.0482, -1.2940,  1.0778, -0.1986])

Output layer weight gradients:
tensor([[ -8.3976, -11.0764,  -2.7311,  -3.7896,  -9.3058,  -2.6694,  -5.9656,
          -7.5355]])

Output layer bias gradient:
tensor([-8.2574])


In [21]:
print("Current first hidden weight:")
print(model[0].weight[0])

print("Gradient:")
print(model[0].weight.grad[0])

print("Learning rate:")
print(optimizer.param_groups[0]["lr"])

Current first hidden weight:
tensor([0.9820], grad_fn=<SelectBackward0>)
Gradient:
tensor([2.2153])
Learning rate:
0.01


In [22]:
optimizer.step()

print("Old weight: 0.9820")
print("New weight:")
print(model[0].weight[0])

Old weight: 0.9820
New weight:
tensor([0.9599], grad_fn=<SelectBackward0>)


In [23]:
prediction = model(x)
loss = loss_fn(prediction, y)

print("New predictions:")
print(prediction)

print("\nNew loss:")
print(loss)

New predictions:
tensor([[ 0.8319],
        [ 0.6980],
        [ 0.5331],
        [ 0.3385],
        [ 0.1409],
        [-0.0189],
        [-0.1787]], grad_fn=<AddmmBackward0>)

New loss:
tensor(25.5847, grad_fn=<MseLossBackward0>)


In [24]:
for step in range(10):

    prediction = model(x)
    loss = loss_fn(prediction, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Step:", step + 1, "Loss:", loss.item())

Step: 1 Loss: 25.584680557250977
Step: 2 Loss: 21.853073120117188
Step: 3 Loss: 18.766647338867188
Step: 4 Loss: 16.137880325317383
Step: 5 Loss: 13.86775016784668
Step: 6 Loss: 11.903849601745605
Step: 7 Loss: 10.217134475708008
Step: 8 Loss: 8.788071632385254
Step: 9 Loss: 7.598465919494629
Step: 10 Loss: 6.627387046813965


In [25]:
print("Predictions:")
print(model(x))

print("\nTargets:")
print(y)

Predictions:
tensor([[5.0909],
        [3.8343],
        [2.6688],
        [1.8776],
        [2.1534],
        [3.4718],
        [4.7902]], grad_fn=<AddmmBackward0>)

Targets:
tensor([[9.],
        [4.],
        [1.],
        [0.],
        [1.],
        [4.],
        [9.]])


In [26]:
loss_history = []

for step in range(90):
    prediction = model(x)
    loss = loss_fn(prediction, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

print("Loss after 90 more steps:", loss_history[-1])

Loss after 90 more steps: 1.019821047782898


In [31]:
prediction = model(x)

print("Predictions:")
for i in range(len(x)):
    print(f"x = {x[i].item():>4.1f}  "
          f"target = {y[i].item():>4.1f}  "
          f"prediction = {prediction[i].item():>6.3f}")

Predictions:
x = -3.0  target =  9.0  prediction =  7.811
x = -2.0  target =  4.0  prediction =  4.985
x = -1.0  target =  1.0  prediction =  2.160
x =  0.0  target =  0.0  prediction =  0.471
x =  1.0  target =  1.0  prediction =  2.002
x =  2.0  target =  4.0  prediction =  4.961
x =  3.0  target =  9.0  prediction =  7.919


In [32]:
test_x = torch.tensor([[-2.5],
                       [-1.5],
                       [ 0.5],
                       [ 1.5],
                       [ 2.5]])

test_prediction = model(test_x)

print("Unseen values:")
for i in range(len(test_x)):
    actual = test_x[i].item() ** 2
    predicted = test_prediction[i].item()

    print(f"x = {test_x[i].item():>4.1f}  "
          f"actual = {actual:>5.2f}  "
          f"prediction = {predicted:>6.3f}")

Unseen values:
x = -2.5  actual =  6.25  prediction =  6.398
x = -1.5  actual =  2.25  prediction =  3.573
x =  0.5  actual =  0.25  prediction =  0.689
x =  1.5  actual =  2.25  prediction =  3.481
x =  2.5  actual =  6.25  prediction =  6.440


In [33]:
test_x = torch.tensor([[-3.0],
                       [-2.0],
                       [-1.0],
                       [ 0.0],
                       [ 1.0],
                       [ 2.0],
                       [ 3.0]])

hidden_before_relu = model[0](test_x)
hidden_after_relu = model[1](hidden_before_relu)

print("Before ReLU:")
print(hidden_before_relu)

print("\nAfter ReLU:")
print(hidden_after_relu)

Before ReLU:
tensor([[-4.5982,  4.4438,  0.1783,  2.0995, -1.7101,  2.4045, -1.4746, -4.1308],
        [-3.3333,  3.0846,  0.2292,  1.0631, -0.9634,  1.5701, -0.9870, -2.8540],
        [-2.0683,  1.7255,  0.2801,  0.0267, -0.2167,  0.7357, -0.4995, -1.5772],
        [-0.8033,  0.3664,  0.3310, -1.0097,  0.5300, -0.0988, -0.0119, -0.3004],
        [ 0.4617, -0.9927,  0.3818, -2.0462,  1.2767, -0.9332,  0.4757,  0.9764],
        [ 1.7267, -2.3518,  0.4327, -3.0826,  2.0234, -1.7677,  0.9632,  2.2532],
        [ 2.9917, -3.7110,  0.4836, -4.1190,  2.7701, -2.6021,  1.4508,  3.5301]],
       grad_fn=<AddmmBackward0>)

After ReLU:
tensor([[0.0000, 4.4438, 0.1783, 2.0995, 0.0000, 2.4045, 0.0000, 0.0000],
        [0.0000, 3.0846, 0.2292, 1.0631, 0.0000, 1.5701, 0.0000, 0.0000],
        [0.0000, 1.7255, 0.2801, 0.0267, 0.0000, 0.7357, 0.0000, 0.0000],
        [0.0000, 0.3664, 0.3310, 0.0000, 0.5300, 0.0000, 0.0000, 0.0000],
        [0.4617, 0.0000, 0.3818, 0.0000, 1.2767, 0.0000, 0.4757, 0.976

In [65]:
test_x = torch.tensor([[-200.0],
                       [-100.0],
                       [ 1000.0],
                       [ 200.0],
                       [ 100.0]])

test_prediction = model(test_x)

print("Unseen values:")
for i in range(len(test_x)):
    actual = test_x[i].item() ** 2
    predicted = test_prediction[i].item()

    print(f"x = {test_x[i].item():>4.1f}  "
          f"actual = {actual:>5.2f}  "
          f"prediction = {predicted:>6.3f}")

Unseen values:
x = -200.0  actual = 40000.00  prediction = 563.347
x = -100.0  actual = 10000.00  prediction = 281.358
x = 1000.0  actual = 1000000.00  prediction = 2957.492
x = 200.0  actual = 40000.00  prediction = 590.733
x = 100.0  actual = 10000.00  prediction = 294.889
